# 04 - Feature Engineering

Builds the modelling table from the cleaned data: removes leakage columns, adds business-driven features, and creates a fixed train/test split.

Output: `Data/Processed/churn_features.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../Data/Processed/telco_churn_clean.csv")
df.shape

(7043, 30)

## Columns removed from the feature set

| Column | Reason |
|---|---|
| `customer_id` | identifier |
| `churn_label` | duplicate of the target |
| `churn_score` | model output built with knowledge of churn (leakage) |
| `churn_reason` | only exists after a customer leaves (leakage) |
| `cltv` | modelled value, kept for business analysis only |
| `city`, `zip_code`, `latitude`, `longitude` | very high cardinality, no clear signal; the geography is revisited in the report |
| `tenure_group` | binned copy of tenure |

In [2]:
drop_cols = ["customer_id", "churn_label", "churn_score", "churn_reason", "cltv",
             "city", "zip_code", "latitude", "longitude", "tenure_group"]
feat = df.drop(columns=drop_cols).copy()
feat.shape

(7043, 20)

## New features

In [3]:
addons = ["online_security", "online_backup", "device_protection", "tech_support", "streaming_tv", "streaming_movies"]
protect = ["online_security", "online_backup", "device_protection", "tech_support"]

# number of add-ons held (0 for customers without internet)
feat["num_addons"] = (df[addons] == "Yes").sum(axis=1)
feat["num_protection_addons"] = (df[protect] == "Yes").sum(axis=1)

# customers who hold neither security nor support
feat["no_security_or_support"] = (
    (df["internet_service"] != "No") & (df["online_security"] != "Yes") & (df["tech_support"] != "Yes")
).astype(int)

feat["has_internet"] = (df["internet_service"] != "No").astype(int)
feat["is_fiber"] = (df["internet_service"] == "Fiber optic").astype(int)
feat["is_month_to_month"] = (df["contract"] == "Month-to-month").astype(int)
feat["is_electronic_check"] = (df["payment_method"] == "Electronic check").astype(int)
feat["is_autopay"] = df["payment_method"].isin(["Bank transfer", "Credit card"]).astype(int)
feat["is_first_year"] = (df["tenure_months"] <= 12).astype(int)
feat["has_family"] = ((df["partner"] == "Yes") | (df["dependents"] == "Yes")).astype(int)

# average spend per month over the customer's life; new customers fall back to the current bill
feat["avg_monthly_spend"] = np.where(df["tenure_months"] > 0,
                                     df["total_charges"] / df["tenure_months"].replace(0, np.nan),
                                     df["monthly_charges"])
# how far the current bill is from the historical average
feat["charge_vs_avg"] = df["monthly_charges"] - feat["avg_monthly_spend"]

# risk interaction found in EDA
feat["m2m_fiber_echeck"] = (feat["is_month_to_month"] & feat["is_fiber"] & feat["is_electronic_check"]).astype(int)

feat.shape

(7043, 33)

## Quick check on the new features

In [4]:
check = []
for col in ["no_security_or_support", "is_first_year", "m2m_fiber_echeck", "is_autopay", "has_family"]:
    g = feat.groupby(col)["churn_value"].agg(["mean", "size"])
    for val, row in g.iterrows():
        check.append({"feature": col, "value": val, "customers": int(row["size"]), "churn_rate_pct": round(row["mean"] * 100, 1)})
pd.DataFrame(check)

,feature,value,customers,churn_rate_pct
0,no_security_or_support,0,4490,13.8
1,no_security_or_support,1,2553,49.0
2,is_first_year,0,4857,17.1
3,is_first_year,1,2186,47.4
4,m2m_fiber_echeck,0,5736,18.8
5,m2m_fiber_echeck,1,1307,60.4
6,is_autopay,0,3977,34.7
7,is_autopay,1,3066,16.0
8,has_family,0,3339,34.4
9,has_family,1,3704,19.4


In [5]:
num_cols = ["tenure_months", "monthly_charges", "total_charges", "num_addons", "num_protection_addons",
            "avg_monthly_spend", "charge_vs_avg"]
feat[num_cols + ["churn_value"]].corr()["churn_value"].drop("churn_value").round(3).sort_values()

tenure_months           -0.352
total_charges           -0.198
num_protection_addons   -0.173
num_addons              -0.088
charge_vs_avg            0.002
monthly_charges          0.193
avg_monthly_spend        0.193

`total_charges` is roughly tenure x average charge, so the numeric features are correlated with each other. This is fine for tree models and for regularised logistic regression, which is how they are used in the next notebook.

## Train / test split

In [6]:
feat["split"] = "train"
train_idx, test_idx = train_test_split(feat.index, test_size=0.2, stratify=feat["churn_value"], random_state=42)
feat.loc[test_idx, "split"] = "test"

print(feat["split"].value_counts())
print(feat.groupby("split")["churn_value"].mean().round(4))

split
train    5634
test     1409
Name: count, dtype: int64
split
test     0.2654
train    0.2654
Name: churn_value, dtype: float64


In [7]:
# keep the id so predictions can be joined back later
out = feat.copy()
out.insert(0, "customer_id", df["customer_id"])
out.to_csv("../Data/Processed/churn_features.csv", index=False)

print(out.shape)
print("No missing values:", out.isna().sum().sum() == 0)
list(out.columns)

(7043, 35)
No missing values: True


['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn_value', 'num_addons', 'num_protection_addons', 'no_security_or_support', 'has_internet', 'is_fiber', 'is_month_to_month', 'is_electronic_check', 'is_autopay', 'is_first_year', 'has_family', 'avg_monthly_spend', 'charge_vs_avg', 'm2m_fiber_echeck', 'split']